# 06 - Matrix Factorization (TruncatedSVD)

Fatora a matriz usuário-item em fatores latentes via `TruncatedSVD`. Funciona como ponte conceitual entre o CF clássico (notebooks 04-05) e os embeddings aprendidos pelo NCF (notebook 07). Ver `docs/NOTEBOOKS.md` (seção 6).

## 0. Configuração Inicial

In [1]:
import json
import pickle
import random
import sys
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml
from sklearn.decomposition import TruncatedSVD

sys.path.insert(0, str(Path("..").resolve()))
from src.evaluation.metrics import evaluate_recommendations, pairs_to_ground_truth
from src.evaluation.ranking import recommendations_from_score_matrix

RANDOM_SEED = 42


def set_seed(seed: int) -> None:
    """Fixa a seed de aleatoriedade para reprodutibilidade."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_SEED)

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models/matrix_factorization")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open("../configs/model_config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
K = config["evaluation"]["k"]
N_COMPONENTS = config["matrix_factorization"]["n_components"]

mlflow.set_tracking_uri(f"file:{(Path('..') / 'mlruns').resolve()}")
mlflow.set_experiment("matrix_factorization")

print(f"K={K}, n_components={N_COMPONENTS}")

K=10, n_components=50


## 1. Carregamento dos Artefatos de Preprocessing

In [2]:
with open(PROCESSED_DIR / "vocabularies.pkl", "rb") as f:
    vocab = pickle.load(f)

interactions_prior = sp.load_npz(PROCESSED_DIR / "interactions_prior.npz")
val_pairs = pd.read_pickle(PROCESSED_DIR / "val_pairs.pkl")
test_pairs = pd.read_pickle(PROCESSED_DIR / "test_pairs.pkl")

n_users, n_items = interactions_prior.shape
val_ground_truth = pairs_to_ground_truth(val_pairs)
test_ground_truth = pairs_to_ground_truth(test_pairs)

print(f"n_users={n_users:,} | n_items={n_items:,}")

n_users=126,408 | n_items=3,000


## 2. Random Search de n_components (Validação)

Busca aleatória sobre os candidatos de `n_components` definidos em `configs/model_config.yaml`, avaliada apenas no split de **validação** (nunca no teste, para não enviesar a escolha do hiperparâmetro). O melhor candidato por `recall@k` é usado na fatoração final da seção 3.

In [3]:
def build_mf_recommendations(
    user_indices: list[int],
    user_factors: np.ndarray,
    item_factors: np.ndarray,
    k: int,
) -> dict[int, list[int]]:
    """Gera recomendacoes via produto interno dos fatores latentes.

    Args:
        user_indices: Lista de user_idx avaliados.
        user_factors: Fatores latentes de usuario (n_users, n_components).
        item_factors: Fatores latentes de item (n_items, n_components).
        k: Tamanho do top-k recomendado.

    Returns:
        Mapeamento user_idx -> lista de item_idx recomendados.
    """
    scores = user_factors[user_indices] @ item_factors.T
    return recommendations_from_score_matrix(user_indices, scores, k)


search_cfg = config["matrix_factorization"]["search"]
candidates = search_cfg["n_components_choices"]
n_trials = min(search_cfg["n_trials"], len(candidates))
search_rng = np.random.default_rng(RANDOM_SEED)
trial_components = search_rng.choice(candidates, size=n_trials, replace=False)

search_results = []
for trial_idx, n_comp in enumerate(trial_components):
    n_comp = int(n_comp)
    trial_svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_SEED)
    trial_user_factors = trial_svd.fit_transform(interactions_prior.astype(np.float32))
    trial_item_factors = trial_svd.components_.T
    trial_recs = build_mf_recommendations(
        list(val_ground_truth.keys()), trial_user_factors, trial_item_factors, K
    )
    trial_metrics = evaluate_recommendations(trial_recs, val_ground_truth, K)
    search_results.append({"n_components": n_comp, **trial_metrics})

    with mlflow.start_run(run_name=f"search_trial_{trial_idx}_ncomp_{n_comp}"):
        mlflow.log_param("n_components", n_comp)
        mlflow.log_metrics(
            {f"val_{name}": value for name, value in trial_metrics.items()}
        )

search_df = pd.DataFrame(search_results).sort_values("recall_at_k", ascending=False)
N_COMPONENTS = int(search_df.iloc[0]["n_components"])
print(search_df)
print(f"Melhor n_components (validacao): {N_COMPONENTS}")

   n_components  precision_at_k  recall_at_k  ndcg_at_k  map_at_k
2           150        0.141765     0.201308   0.205638  0.114846
1           100        0.137820     0.191130   0.200765  0.112031
3            50        0.128532     0.170993   0.189012  0.104255
0            20        0.113243     0.147857   0.176506  0.097316
Melhor n_components (validacao): 150


## 3. Fatoração Final via TruncatedSVD (Melhor n_components)

In [4]:
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_SEED)
user_factors = svd.fit_transform(interactions_prior.astype(np.float32))
item_factors = svd.components_.T

explained_variance = float(svd.explained_variance_ratio_.sum())
print(f"user_factors={user_factors.shape} | item_factors={item_factors.shape}")
print(f"explained_variance_ratio={explained_variance:.4f}")

user_factors=(126408, 150) | item_factors=(3000, 150)
explained_variance_ratio=0.3253


## 4. Geração de Recomendações (Produto Interno dos Fatores Latentes)

In [5]:
val_recommendations = build_mf_recommendations(
    list(val_ground_truth.keys()), user_factors, item_factors, K
)
test_recommendations = build_mf_recommendations(
    list(test_ground_truth.keys()), user_factors, item_factors, K
)

## 5. Avaliação

In [6]:
val_metrics = evaluate_recommendations(val_recommendations, val_ground_truth, K)
test_metrics = evaluate_recommendations(test_recommendations, test_ground_truth, K)

print("Validação:", val_metrics)
print("Teste:", test_metrics)

Validação: {'precision_at_k': 0.14176467485892097, 'recall_at_k': 0.20130802956816043, 'ndcg_at_k': 0.2056384875405363, 'map_at_k': 0.11484570651007679}
Teste: {'precision_at_k': 0.139848117287206, 'recall_at_k': 0.20171414600131654, 'ndcg_at_k': 0.20324615600654183, 'map_at_k': 0.1136531168166384}


## 6. Persistência e Rastreamento no MLflow

In [7]:
np.save(MODELS_DIR / "user_factors.npy", user_factors)
np.save(MODELS_DIR / "item_factors.npy", item_factors)

with open(PROCESSED_DIR / "split_meta.json", encoding="utf-8") as f:
    split_meta = json.load(f)

metrics_payload = {
    "k": K,
    "n_components": N_COMPONENTS,
    "explained_variance_ratio": explained_variance,
    "search_results": search_results,
    "validation": val_metrics,
    "test": test_metrics,
}
with open(MODELS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

with mlflow.start_run(run_name="matrix_factorization_v1"):
    mlflow.log_params(
        {
            "k": K,
            "n_components": N_COMPONENTS,
            "dataset_hash": split_meta["dataset_hash"],
            "selected_via_random_search": True,
        }
    )
    mlflow.log_metric("explained_variance_ratio", explained_variance)
    mlflow.log_metrics({f"val_{name}": value for name, value in val_metrics.items()})
    mlflow.log_metrics({f"test_{name}": value for name, value in test_metrics.items()})
    mlflow.log_artifact(str(MODELS_DIR / "metrics.json"))

print("Matrix Factorization avaliado e rastreado no MLflow.")

Matrix Factorization avaliado e rastreado no MLflow.
